# TAVIX AI: free QLoRA fine-tune (Kaggle T4)
Before running: Settings > Accelerator = **GPU T4 x2** (not P100), Internet = **On**, add your dataset (train.jsonl + valid.jsonl), add Kaggle secret `HF_TOKEN` (Hugging Face write token). Edit the 3 values in cell 2.

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade trl datasets

In [ ]:
DATA_DIR = "/kaggle/input/YOUR-DATASET-NAME"      # folder shown in the right-hand 'Input' panel
HF_REPO  = "YOUR-HF-USERNAME/tavix-7b"           # the private repo that will hold your merged model
BASE     = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAXLEN   = 4096

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAXLEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl", "validation": f"{DATA_DIR}/valid.jsonl"})
ds = ds.map(lambda ex: {"text": tok.apply_chat_template(ex["messages"], tokenize=False)})
# Drop examples longer than the context window: truncated websites would teach the model to stop mid-file
before = {k: len(v) for k, v in ds.items()}
ds = ds.filter(lambda ex: len(tok(ex["text"])["input_ids"]) <= MAXLEN)
print("kept", {k: len(v) for k, v in ds.items()}, "of", before)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds["train"], eval_dataset=ds["validation"],
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAXLEN, per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=5, eval_strategy="steps", eval_steps=50, output_dir="out", report_to="none"))
trainer.train()

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
model.push_to_hub_merged(HF_REPO, tok, save_method="merged_16bit", token=HF_TOKEN, private=True)
print("Pushed", HF_REPO)

If a cell fails with an argument name error (e.g. `max_seq_length`), read the message and rename it: library versions change often.